In [1]:
import pandas as pd
import glob


In [11]:
import glob
import pandas as pd

PATH = r"C:/Users/alexa/OneDrive/Desktop/IDX Internship/raw/"

files = glob.glob(PATH + "CRMLSListing*")

print(f"Number of monthly files found: {len(files)}")

row_counts = []
for f in files:
    temp = pd.read_csv(f, encoding='latin1')
    row_counts.append((f, len(temp)))

print("\nRow counts BEFORE concatenation:")
for fname, count in row_counts:
    print(f"{fname}: {count:,} rows")

df_listed = pd.concat(
    [pd.read_csv(f, encoding='latin1') for f in files],
    ignore_index=True
)

print(f"\nTotal rows AFTER concatenation: {len(df_listed):,}")

df_res = df_listed[df_listed["PropertyType"] == "Residential"]

print(f"Total rows AFTER Residential filter: {len(df_res):,}")

df_res.to_csv("combined_listed.csv", index=False)
print("\nSaved filtered dataset to combined_listed.csv")


Number of monthly files found: 27

Row counts BEFORE concatenation:
C:/Users/alexa/OneDrive/Desktop/IDX Internship/raw\CRMLSListing202401.csv: 27,454 rows
C:/Users/alexa/OneDrive/Desktop/IDX Internship/raw\CRMLSListing202402.csv: 27,447 rows
C:/Users/alexa/OneDrive/Desktop/IDX Internship/raw\CRMLSListing202403.csv: 32,282 rows
C:/Users/alexa/OneDrive/Desktop/IDX Internship/raw\CRMLSListing202404.csv: 36,503 rows
C:/Users/alexa/OneDrive/Desktop/IDX Internship/raw\CRMLSListing202405.csv: 38,796 rows
C:/Users/alexa/OneDrive/Desktop/IDX Internship/raw\CRMLSListing202406.csv: 35,893 rows
C:/Users/alexa/OneDrive/Desktop/IDX Internship/raw\CRMLSListing202407.csv: 36,340 rows
C:/Users/alexa/OneDrive/Desktop/IDX Internship/raw\CRMLSListing202408.csv: 35,305 rows
C:/Users/alexa/OneDrive/Desktop/IDX Internship/raw\CRMLSListing202409.csv: 34,625 rows
C:/Users/alexa/OneDrive/Desktop/IDX Internship/raw\CRMLSListing202410.csv: 34,730 rows
C:/Users/alexa/OneDrive/Desktop/IDX Internship/raw\CRMLSListin

In [14]:
df_res.shape
df_res.info()


<class 'pandas.core.frame.DataFrame'>
Index: 534610 entries, 2 to 845400
Data columns (total 84 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   OriginalListPrice             533842 non-null  float64
 1   ListingKey                    534610 non-null  int64  
 2   ListAgentEmail                493182 non-null  object 
 3   CloseDate                     169865 non-null  object 
 4   ClosePrice                    150269 non-null  float64
 5   ListAgentFirstName            530455 non-null  object 
 6   ListAgentLastName             534570 non-null  object 
 7   Latitude                      454543 non-null  float64
 8   Longitude                     454543 non-null  float64
 9   UnparsedAddress               533969 non-null  object 
 10  PropertyType                  534610 non-null  object 
 11  LivingArea                    534071 non-null  float64
 12  ListPrice                     534610 non-null  fl

In [17]:

print("Initial row count:", len(df_res))
print("Initial column count:", df_res.shape[1])

# 1. DOCUMENT UNIQUE PROPERTY TYPES
print("\nUnique Property Types:")
print(df_listed["PropertyType"].unique())

# 2. NULL COUNT SUMMARY TABLE
null_summary = df_res.isnull().sum().to_frame(name="NullCount")
null_summary["NullPercent"] = (null_summary["NullCount"] / len(df_res)) * 100

print("\nNull Count Summary Table:")
print(null_summary)

# 3. FLAG COLUMNS ABOVE 90% NULL
high_null_cols = null_summary[null_summary["NullPercent"] > 90].index.tolist()

print("\nColumns ABOVE 90% null:")
for col in high_null_cols:
    print(f"- {col}")

# 4. REMOVE COLUMNS ABOVE 90% NULL
df_filtered = df_res.drop(columns=high_null_cols)
print(f"\nColumn count AFTER removing >90% null columns: {df_filtered.shape[1]}")

# 5. NUMERIC DISTRIBUTION SUMMARY
#    For ClosePrice, LivingArea, DaysOnMarket
numeric_cols = ["ClosePrice", "LivingArea", "DaysOnMarket"]

print("\nNumeric Distribution Summary:")
for col in numeric_cols:
    if col in df_filtered.columns:
        print(f"\n--- {col} ---")
        print(df_filtered[col].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))
    else:
        print(f"\n--- {col} NOT FOUND IN DATASET ---")

# 6. SAVE FILTERED DATASET
df_filtered.to_csv("filtered_week2_3_listed_output.csv", index=False)
print("\nSaved cleaned dataset to filtered_week2_3_listed_output.csv")


Initial row count: 534610
Initial column count: 84

Unique Property Types:
['ManufacturedInPark' 'CommercialSale' 'Residential' 'ResidentialLease'
 'Land' 'ResidentialIncome' 'CommercialLease' 'BusinessOpportunity']

Null Count Summary Table:
                              NullCount  NullPercent
OriginalListPrice                   768     0.143656
ListingKey                            0     0.000000
ListAgentEmail                    41428     7.749200
CloseDate                        364745    68.226371
ClosePrice                       384341    71.891846
...                                 ...          ...
BuyerOfficeName.1                373506    69.865135
AssociationFee                   127947    23.932773
LotSizeSquareFeet                 43411     8.120125
MiddleOrJuniorSchoolDistrict     534610   100.000000
UnparsedAddress.1                   641     0.119900

[84 rows x 2 columns]

Columns ABOVE 90% null:
- FireplacesTotal
- AboveGradeFinishedArea
- TaxAnnualAmount
- BuilderNam

In [18]:

url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US"

mortgage = pd.read_csv(url, parse_dates=['observation_date'])
mortgage.columns = ['date', 'rate_30yr_fixed']

mortgage['year_month'] = mortgage['date'].dt.to_period('M')

mortgage_monthly = (
    mortgage.groupby('year_month')['rate_30yr_fixed']
    .mean()
    .reset_index()
)

df_filtered['year_month'] = pd.to_datetime(
    df_filtered['ListingContractDate']
).dt.to_period('M')

listed_with_rates = df_filtered.merge(
    mortgage_monthly, on='year_month', how='left'
)

null_rates = listed_with_rates['rate_30yr_fixed'].isnull().sum()
print("Listed rows with NULL mortgage rate:", null_rates)

listed_with_rates.to_csv("listed_with_mortgage_rates.csv", index=False)
print("Saved listed_with_mortgage_rates.csv")

Listed rows with NULL mortgage rate: 0
Saved listed_with_mortgage_rates.csv


In [19]:

# 1. LOAD EXISTING LISTINGS CSV (EDIT IN PLACE)

print("Initial row count:", len(listed_with_rates))
print("Initial column count:", listed_with_rates.shape[1])

# 2. DATA TYPE CONFIRMATION

print("\nData types BEFORE cleaning:")
print(listed_with_rates.dtypes)

# 3. DATE CONSISTENCY CHECKS
date_cols = ["ListingContractDate", "CloseDate"]

for col in date_cols:
    if col in listed_with_rates.columns:
        listed_with_rates[col] = pd.to_datetime(listed_with_rates[col], errors="coerce")

print("\nInvalid date counts:")
for col in date_cols:
    if col in listed_with_rates.columns:
        invalid = listed_with_rates[col].isnull().sum()
        print(f"{col}: {invalid} invalid entries")


# 4. GEOGRAPHIC DATA QUALITY CHECK

lat_col = "Latitude"
lon_col = "Longitude"

invalid_lat = listed_with_rates[(listed_with_rates[lat_col] < -90) | (listed_with_rates[lat_col] > 90)].shape[0] if lat_col in listed_with_rates else 0
invalid_lon = listed_with_rates[(listed_with_rates[lon_col] < -180) | (listed_with_rates[lon_col] > 180)].shape[0] if lon_col in listed_with_rates else 0

print("\nGeographic Data Quality Summary:")
print(f"Invalid Latitude values: {invalid_lat}")
print(f"Invalid Longitude values: {invalid_lon}")

if lat_col in listed_with_rates and lon_col in listed_with_rates:
    df = listed_with_rates[
        (listed_with_rates[lat_col].between(-90, 90)) &
        (listed_with_rates[lon_col].between(-180, 180))
    ]

print("Rows AFTER removing invalid coordinates:", len(df))

# 5. FINAL DATA TYPE CHECK

print("\nData types AFTER cleaning:")
print(listed_with_rates.dtypes)

# 6. SAVE CLEANED, ANALYSIS-READY DATASET

listed_with_rates.to_csv("listings_cleaned_final.csv", index=False)

print("\nSaved listings_cleaned_final.csv")


Initial row count: 534610
Initial column count: 73

Data types BEFORE cleaning:
OriginalListPrice      float64
ListingKey               int64
ListAgentEmail          object
CloseDate               object
ClosePrice             float64
                       ...    
AssociationFee         float64
LotSizeSquareFeet      float64
UnparsedAddress.1       object
year_month           period[M]
rate_30yr_fixed        float64
Length: 73, dtype: object

Invalid date counts:
ListingContractDate: 0 invalid entries
CloseDate: 364745 invalid entries

Geographic Data Quality Summary:
Invalid Latitude values: 3
Invalid Longitude values: 1
Rows AFTER removing invalid coordinates: 454539

Data types AFTER cleaning:
OriginalListPrice           float64
ListingKey                    int64
ListAgentEmail               object
CloseDate            datetime64[ns]
ClosePrice                  float64
                          ...      
AssociationFee              float64
LotSizeSquareFeet           float64
Unpar

In [2]:
import pandas as pd

df = pd.read_csv("../raw/listings_cleaned_final.csv")

df["PriceRatio"] = df["ClosePrice"] / df["OriginalListPrice"]

df["CloseToOriginalListRatio"] = df["ClosePrice"] / df["OriginalListPrice"]

df["PPSF"] = df["ClosePrice"] / df["LivingArea"]

df["DaysOnMarket"] = df["DaysOnMarket"]

df["CloseDate"] = pd.to_datetime(df["CloseDate"])
df["CloseYear"] = df["CloseDate"].dt.year
df["CloseMonth"] = df["CloseDate"].dt.month
df["YrMo"] = df["CloseDate"].dt.to_period("M").astype(str)

df["ListingToContractDays"] = (
    pd.to_datetime(df["PurchaseContractDate"]) -
    pd.to_datetime(df["ListingContractDate"])
).dt.days

df["ContractToCloseDays"] = (
    pd.to_datetime(df["CloseDate"]) -
    pd.to_datetime(df["PurchaseContractDate"])
).dt.days

sample_output = df[[
    "ClosePrice", "OriginalListPrice", "LivingArea",
    "PriceRatio", "CloseToOriginalListRatio", "PPSF",
    "DaysOnMarket", "YrMo",
    "ListingToContractDays", "ContractToCloseDays",
    "PropertyType", "CountyOrParish"
]].head(10)

print("\n=== SAMPLE OUTPUT TABLE ===")
print(sample_output)

segment_summary = df.groupby("PropertyType").agg({
    "ClosePrice": "median",
    "PPSF": "median",
    "DaysOnMarket": "median",
    "PriceRatio": "median",
    "ListingToContractDays": "median",
    "ContractToCloseDays": "median"
}).reset_index()

print("\n=== SEGMENT SUMMARY BY PROPERTY TYPE ===")
print(segment_summary)

sample_output.to_csv("week6_sample_output.csv", index=False)
segment_summary.to_csv("week6_segment_summary.csv", index=False)

print("\nWeek 6 feature engineering complete.")


C:\Users\alexa\AppData\Local\Temp\ipykernel_27736\2190451260.py:3: DtypeWarning: Columns (2,39) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../raw/listings_cleaned_final.csv")



=== SAMPLE OUTPUT TABLE ===
   ClosePrice  OriginalListPrice  LivingArea  PriceRatio  \
0         NaN          1340000.0      1301.0         NaN   
1         NaN          2500000.0      2788.0         NaN   
2         NaN          3150000.0      3250.0         NaN   
3         NaN          3090000.0      7456.0         NaN   
4         NaN         12725000.0      2321.0         NaN   
5         NaN           665000.0      1487.0         NaN   
6         NaN          1125000.0      1750.0         NaN   
7         NaN          1849000.0      3152.0         NaN   
8         NaN          1389000.0      2282.0         NaN   
9   1262555.0          1249888.0      1899.0    1.010135   

   CloseToOriginalListRatio        PPSF  DaysOnMarket     YrMo  \
0                       NaN         NaN           127      NaT   
1                       NaN         NaN             1      NaT   
2                       NaN         NaN             1      NaT   
3                       NaN         NaN       